[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/duckdb-certified/notebooks/day-04-python-api.ipynb#scrollTo=a1b2c3d4)

---
# Day 4 · The Python API — Connecting, Executing, and Fetching
**certified-journeys / duckdb-certified** · Practice · Python API

> **Goal for today:** By the end of this notebook you can open DuckDB connections, run parameterised queries safely, retrieve results in multiple formats, register Python objects as virtual tables, and wrap everything in a clean helper function.


In [ ]:
%pip install -q duckdb pandas pyarrow


## Step 1 · In-memory vs file-backed connections

DuckDB supports two connection modes:

| Mode | How to open | Persistence | Typical use |
|------|-------------|-------------|-------------|
| In-memory | `duckdb.connect()` or `duckdb.connect(':memory:')` | Lost on close | Scripts, notebooks, tests |
| File-backed | `duckdb.connect('path/to/file.db')` | Survives restarts | Long-lived datasets, CI pipelines |

An in-memory connection is the default. A file-backed connection creates the file if it does not exist.
Only **one write connection** may be open on the same file at a time; multiple **read-only** connections are fine.


In [ ]:
import duckdb
import os

# --- In-memory connection (default) ---
mem_con = duckdb.connect()          # identical to duckdb.connect(':memory:')
mem_con.execute("CREATE TABLE nums AS SELECT range(1, 11) AS n")
print("In-memory rows:", mem_con.execute("SELECT COUNT(*) FROM nums").fetchone()[0])

# --- File-backed connection ---
DB_PATH = "/tmp/demo_day04.duckdb"
file_con = duckdb.connect(DB_PATH)  # creates the file if absent
file_con.execute("CREATE OR REPLACE TABLE nums AS SELECT range(1, 11) AS n")
print("File-backed rows:", file_con.execute("SELECT COUNT(*) FROM nums").fetchone()[0])
print("File exists:", os.path.exists(DB_PATH))

# Always close connections when done to release file locks
mem_con.close()
file_con.close()


### What just happened?
- **`duckdb.connect()`** with no argument gives a fresh in-memory database — data vanishes when the connection is closed.
- **`duckdb.connect('path.duckdb')`** creates (or opens) a persistent file; calling `.close()` flushes writes.
- `CREATE OR REPLACE TABLE` is a safe pattern when the file already contains data from a previous run.
- **One write lock per file** — trying to open the same file twice for writing will raise a `duckdb.IOException`.


## Step 2 · Parameterised queries — safe and fast

Always use `?` placeholders instead of f-strings to prevent SQL injection and avoid re-parsing the query plan inside loops.

```
# Unsafe — never do this
con.execute(f"SELECT * FROM orders WHERE status = '{user_input}'")

# Safe — use positional parameters
con.execute("SELECT * FROM orders WHERE status = ?", [user_input])
```

DuckDB also supports **named parameters** using `$name` syntax for readability in complex queries.


In [ ]:
import duckdb

con = duckdb.connect()

# Build a small dataset
con.execute("""
    CREATE TABLE products (
        id      INTEGER,
        name    VARCHAR,
        price   DOUBLE,
        category VARCHAR
    )
""")
con.executemany(
    "INSERT INTO products VALUES (?, ?, ?, ?)",
    [
        (1, 'Widget A', 9.99,  'tools'),
        (2, 'Gadget B', 49.99, 'electronics'),
        (3, 'Doohickey', 4.50, 'tools'),
        (4, 'Thingamajig', 99.00, 'electronics'),
        (5, 'Whatsit',   12.00, 'tools'),
    ]
)

# Single-parameter query
cat = 'tools'
rows = con.execute(
    "SELECT name, price FROM products WHERE category = ? ORDER BY price",
    [cat]
).fetchall()
print(f"Products in '{cat}':")
for name, price in rows:
    print(f"  {name}: ${price:.2f}")

# Multi-parameter query
min_price, max_price = 5.0, 55.0
rows2 = con.execute(
    "SELECT name, price, category FROM products WHERE price BETWEEN ? AND ? ORDER BY price",
    [min_price, max_price]
).fetchall()
print(f"\nProducts between ${min_price} and ${max_price}:")
for r in rows2:
    print(f"  {r}")

con.close()


### What just happened?
- **`executemany`** batches inserts efficiently; each row is a tuple matching the `?` placeholder count.
- **Positional `?`** placeholders are bound in order from the list — DuckDB caches the prepared plan.
- `fetchall()` returns a list of tuples — lightweight, good for small result sets.
- **`BETWEEN ? AND ?`** uses two parameters; the list must match the placeholder count exactly or DuckDB raises `InvalidInputException`.


## Step 3 · Fetching results in multiple formats

DuckDB result objects expose several fetch methods — pick the one that matches your downstream code:

| Method | Returns | Best for |
|--------|---------|----------|
| `.fetchone()` | Single tuple or `None` | Scalar lookups |
| `.fetchall()` | List of tuples | Small result sets |
| `.fetchdf()` / `.df()` | `pandas.DataFrame` | Data analysis |
| `.fetchnumpy()` | Dict of NumPy arrays | Numeric processing |
| `.fetch_arrow_table()` / `.arrow()` | `pyarrow.Table` | Zero-copy pipelines |
| `.fetchmany(n)` | List of n tuples | Streaming large results |


In [ ]:
import duckdb
import pyarrow as pa

con = duckdb.connect()
con.execute("""
    CREATE TABLE sales AS
    SELECT
        i                          AS id,
        'Product ' || i            AS product,
        ROUND(RANDOM() * 500, 2)   AS revenue,
        DATE '2024-01-01' + INTERVAL (i % 30) DAY AS sale_date
    FROM range(1, 51) t(i)
""")

# 1 — fetchone: single row / scalar
total = con.execute("SELECT SUM(revenue) FROM sales").fetchone()[0]
print(f"Total revenue: ${total:,.2f}")

# 2 — fetchall: list of tuples
top3 = con.execute("SELECT product, revenue FROM sales ORDER BY revenue DESC LIMIT 3").fetchall()
print("\nTop 3 by revenue (tuples):", top3)

# 3 — fetchdf: pandas DataFrame
df = con.execute("SELECT product, revenue, sale_date FROM sales WHERE revenue > 400").fetchdf()
print(f"\nHigh-value sales (DataFrame, {len(df)} rows):")
print(df.head(3).to_string(index=False))

# 4 — fetch_arrow_table: pyarrow Table (zero-copy on supported types)
arrow_tbl = con.execute("SELECT id, revenue FROM sales LIMIT 5").fetch_arrow_table()
print(f"\nArrow table schema: {arrow_tbl.schema}")
print("Arrow column 'revenue':", arrow_tbl.column('revenue').to_pylist())

con.close()


### What just happened?
- **`fetchone()[0]`** extracts a scalar from the single-column, single-row result — compact for aggregation checks.
- **`fetchdf()`** is an alias for `.df()` — both return a Pandas DataFrame; column names come from the query aliases.
- **`fetch_arrow_table()`** returns a `pyarrow.Table` without serialising data through Python objects — ideal for passing to Arrow-aware libraries (Polars, Parquet writers, Flight servers).
- **`RANDOM()`** in a `CREATE TABLE AS SELECT` seeds each run differently — results will vary between runs.


## Step 4 · The `duckdb.sql()` shortcut

For one-shot queries where you don't want to manage a connection object, use the module-level `duckdb.sql()` function.
It uses an implicit global in-memory connection that persists for the process lifetime.

> **When to use it:** quick explorations, REPL sessions, scripts that create-read-dispose in one go.
> **When NOT to use it:** production pipelines or concurrent code — the shared global state can cause surprising interactions.

Reference: [DuckDB Python DB-API](https://duckdb.org/docs/api/python/dbapi)


In [ ]:
import duckdb

# No connection object needed — uses the implicit default connection
result = duckdb.sql("SELECT 40 + 2 AS answer")
print("Direct SQL result:", result.fetchone())

# Chain methods: sql() → df() in one expression
df = duckdb.sql("""
    SELECT
        unnest(['Alice', 'Bob', 'Carol', 'Dave']) AS name,
        unnest([88, 72, 95, 61])                  AS score
""").df()
print("\nScores DataFrame:")
print(df)

# Read a remote (or local) Parquet / CSV without a connection
# Here we write a tiny CSV first so the notebook is self-contained
import tempfile, os
csv_path = os.path.join(tempfile.gettempdir(), 'sample.csv')
with open(csv_path, 'w') as f:
    f.write("city,pop\nNew York,8336817\nLos Angeles,3979576\nChicago,2693976\n")

city_df = duckdb.sql(f"SELECT city, pop FROM read_csv('{csv_path}') ORDER BY pop DESC").df()
print("\nCities:")
print(city_df)


### What just happened?
- **`duckdb.sql()`** returns a `DuckDBPyRelation` — a lazy relational object you chain `.fetchall()`, `.df()`, `.arrow()` etc. on.
- **`unnest()`** turns a list literal into rows — useful for inline test data.
- The **implicit connection** remembers tables created via `duckdb.sql()` within the same process — useful but can cause hidden state issues in longer scripts.
- `read_csv()` and `read_parquet()` work the same way without any explicit connection.


## Step 5 · Registering Python objects as virtual tables

`con.register(name, python_object)` exposes a Python list of dicts (or a Pandas/Polars/Arrow object) as a queryable SQL table — no COPY or INSERT required.

Supported objects: `list[dict]`, `pandas.DataFrame`, `polars.DataFrame`, `pyarrow.Table`, `pyarrow.RecordBatch`.

The table is **virtual** — it does not persist after the connection is closed.


In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect()

# --- Register a list of dicts ---
employees = [
    {'id': 1, 'name': 'Alice', 'dept': 'Engineering', 'salary': 95000},
    {'id': 2, 'name': 'Bob',   'dept': 'Marketing',   'salary': 72000},
    {'id': 3, 'name': 'Carol', 'dept': 'Engineering', 'salary': 110000},
    {'id': 4, 'name': 'Dave',  'dept': 'Marketing',   'salary': 68000},
    {'id': 5, 'name': 'Eve',   'dept': 'Engineering', 'salary': 88000},
]
con.register('employees', employees)   # name the virtual table

result = con.execute("""
    SELECT dept, AVG(salary) AS avg_salary, COUNT(*) AS headcount
    FROM employees
    GROUP BY dept
    ORDER BY avg_salary DESC
""").fetchdf()
print("Dept summary from list-of-dicts:")
print(result)

# --- Register a Pandas DataFrame ---
df_budgets = pd.DataFrame({
    'dept':   ['Engineering', 'Marketing'],
    'budget': [500_000, 200_000],
})
con.register('budgets', df_budgets)

joined = con.execute("""
    SELECT e.dept, e.avg_salary, b.budget,
           ROUND(e.avg_salary * e.headcount / b.budget * 100, 1) AS pct_of_budget
    FROM (
        SELECT dept, AVG(salary) AS avg_salary, COUNT(*) AS headcount
        FROM employees GROUP BY dept
    ) e
    JOIN budgets b USING (dept)
""").fetchdf()
print("\nBudget utilisation:")
print(joined)

con.close()


### What just happened?
- **`con.register('name', obj)`** maps a Python object to a SQL view name — DuckDB auto-detects the schema from the data.
- **List of dicts** gets its column names from the dict keys; all rows must share the same key set.
- Registered DataFrames and lists are **read via memory pointers** — no data is copied into DuckDB storage.
- **JOINs between registered objects** work exactly like JOINs between persistent tables — the planner treats them identically.
- Call `con.unregister('name')` to remove the virtual table before closing if you need to reuse the name.


## Step 6 · `duckdb.execute()` with the DB-API interface

DuckDB implements [PEP 249 (DB-API 2.0)](https://peps.python.org/pep-0249/), the same interface used by `sqlite3`, `psycopg2`, and most SQL drivers.
This means you can swap DuckDB into any code that was written for SQLite with minimal changes.

Key DB-API concepts:
- **Connection** — manages the database session.
- **Cursor** — created from a connection to execute statements and iterate over results.
- DuckDB's connection object acts as its own cursor (convenience shortcut), but you can also call `con.cursor()`.


In [ ]:
import duckdb

con = duckdb.connect()

# Create table using DB-API style execute
con.execute("""
    CREATE TABLE events (
        event_id   INTEGER,
        event_type VARCHAR,
        ts         TIMESTAMP,
        payload    VARCHAR
    )
""")

# executemany for bulk inserts (DB-API standard)
rows = [
    (1, 'click',   '2024-03-01 10:00:00', '{"button": "buy"}'),
    (2, 'view',    '2024-03-01 10:01:00', '{"page": "/checkout"}'),
    (3, 'click',   '2024-03-01 10:02:00', '{"button": "confirm"}'),
    (4, 'purchase','2024-03-01 10:03:00', '{"amount": 49.99}'),
    (5, 'view',    '2024-03-01 10:04:00', '{"page": "/thank-you"}'),
]
con.executemany("INSERT INTO events VALUES (?, ?, ?, ?)", rows)

# Explicit cursor for DB-API purity
cur = con.cursor()
cur.execute("SELECT event_type, COUNT(*) AS cnt FROM events GROUP BY event_type ORDER BY cnt DESC")

# Cursor description gives column metadata (DB-API standard)
print("Columns:", [col[0] for col in cur.description])
print("Results:")
for row in cur.fetchall():
    print(" ", row)

cur.close()
con.close()


### What just happened?
- **`con.cursor()`** returns a `DuckDBPyCursor` — a standard DB-API cursor; `cur.description` lists `(name, type_code, …)` tuples.
- **`executemany`** on a cursor is more memory-efficient than looping `execute` — DuckDB sends the batch in one round-trip.
- The cursor supports **iteration** (`for row in cur`) after `execute` — useful for streaming large results without `fetchall` loading everything into memory.
- **DB-API compatibility** means existing SQLAlchemy or pandas `read_sql` integrations work with the DuckDB connection.


## Step 7 · Context managers and safe connection handling

DuckDB connections implement the context manager protocol (`__enter__` / `__exit__`), so you can use `with` blocks.
This guarantees the connection is closed — and file locks are released — even if an exception occurs mid-query.


In [ ]:
import duckdb
import pandas as pd

# Pattern 1: context manager (preferred for scripts)
with duckdb.connect() as con:
    con.execute("CREATE TABLE t AS SELECT range(5) AS n")
    rows = con.execute("SELECT n * n AS n_squared FROM t").fetchall()
print("Squares:", rows)
# con is automatically closed here

# Pattern 2: try/finally (explicit, identical semantics)
con2 = duckdb.connect()
try:
    df = con2.execute("SELECT 'hello' AS greeting, 42 AS answer").fetchdf()
    print(df)
finally:
    con2.close()

# Pattern 3: using read_only flag for concurrent read scenarios
DB_PATH = "/tmp/demo_day04.duckdb"
# Write connection
write_con = duckdb.connect(DB_PATH)
write_con.execute("CREATE OR REPLACE TABLE log AS SELECT range(3) AS i")
write_con.close()  # must close before opening read-only

# Read-only — safe for concurrent readers (e.g., multiple Jupyter kernels)
with duckdb.connect(DB_PATH, read_only=True) as ro:
    print("Read-only rows:", ro.execute("SELECT * FROM log").fetchall())


### What just happened?
- **`with duckdb.connect() as con:`** automatically calls `con.close()` on exit — the preferred pattern for notebooks and scripts.
- **`read_only=True`** allows multiple concurrent readers on the same file — DuckDB uses a shared-memory lock rather than an exclusive write lock.
- **`try/finally`** is the explicit equivalent; useful when you need the connection object after a potential error for cleanup.
- Closing a write connection **flushes WAL (Write-Ahead Log)** to the file — without closing, partial writes may not be visible to other processes.


## Step 8 · Building a reusable query helper

A single helper function encapsulates the boilerplate: open connection → execute → fetch → close.
It returns a Pandas DataFrame for easy downstream use and accepts optional parameters for safe injection prevention.


In [ ]:
import duckdb
import pandas as pd
from typing import Any, Optional


def query_duckdb(
    sql: str,
    params: Optional[list] = None,
    db_path: str = ":memory:",
    registered: Optional[dict] = None,
) -> pd.DataFrame:
    """
    Execute a SQL query against a DuckDB database and return a DataFrame.

    Args:
        sql:        The SQL query string; use ? for positional parameters.
        params:     Optional list of values bound to ? placeholders.
        db_path:    Path to a DuckDB file, or ':memory:' (default).
        registered: Optional dict of {name: python_object} to register
                    as virtual tables before executing the query.

    Returns:
        A pandas.DataFrame with the query result.

    Raises:
        duckdb.Error: on any SQL or connection error (propagated to caller).
    """
    con = duckdb.connect(db_path)
    try:
        if registered:
            for name, obj in registered.items():
                con.register(name, obj)
        result = con.execute(sql, params or []).fetchdf()
    finally:
        con.close()   # always closes, even if execute() raises
    return result


# --- Demo 1: basic query on in-memory table ---
# We create the table inside a separate connection first
# to show the helper works on file-backed DBs too
DB_PATH = "/tmp/demo_helper.duckdb"
setup_con = duckdb.connect(DB_PATH)
setup_con.execute("""
    CREATE OR REPLACE TABLE orders AS
    SELECT i AS order_id,
           CASE WHEN i % 3 = 0 THEN 'pending'
                WHEN i % 3 = 1 THEN 'shipped'
                ELSE 'delivered' END AS status,
           ROUND(RANDOM() * 200 + 10, 2) AS amount
    FROM range(1, 21) t(i)
""")
setup_con.close()

# Use the helper
df_shipped = query_duckdb(
    "SELECT order_id, amount FROM orders WHERE status = ? ORDER BY amount DESC",
    params=['shipped'],
    db_path=DB_PATH,
)
print(f"Shipped orders ({len(df_shipped)} rows):")
print(df_shipped.head())

# --- Demo 2: query over a registered Python object ---
inventory = [
    {'sku': 'A001', 'qty': 150, 'reorder_level': 100},
    {'sku': 'B002', 'qty': 45,  'reorder_level': 50},
    {'sku': 'C003', 'qty': 200, 'reorder_level': 75},
]
df_reorder = query_duckdb(
    "SELECT sku, qty, reorder_level FROM inventory WHERE qty < reorder_level",
    registered={'inventory': inventory},
)
print(f"\nItems needing reorder: {len(df_reorder)}")
print(df_reorder)


### What just happened?
- The helper uses **`try/finally`** to guarantee `.close()` runs regardless of success or error.
- **`params or []`** ensures we never pass `None` to `execute()` — DuckDB expects a list even when there are no parameters.
- The **`registered` dict** lets callers inject Python objects without modifying the helper; the objects live only for the duration of the call.
- Returning a **DataFrame** makes the helper composable with the rest of a pandas/Polars pipeline.


In [ ]:
# Challenge: Build a batch-query function
#
# Write a function `batch_query(sql, param_batches, db_path)` that:
#   1. Opens ONE connection to db_path
#   2. Executes `sql` for each list of params in `param_batches`
#   3. Concatenates the results into a single DataFrame
#   4. Closes the connection safely
#
# Hint: use pd.concat() on a list of DataFrames.
# Hint: reuse the connection across iterations — don't reopen per batch.
#
# Test it with:
#   DB_PATH = '/tmp/demo_helper.duckdb'   (created above)
#   sql = "SELECT order_id, status, amount FROM orders WHERE status = ?"
#   batches = [['shipped'], ['pending'], ['delivered']]
#
# Expected: a DataFrame with all 20 rows, all statuses combined.

import duckdb
import pandas as pd

def batch_query(sql: str, param_batches: list, db_path: str = ":memory:") -> pd.DataFrame:
    # Your solution here
    pass


# Uncomment to test:
# DB_PATH = '/tmp/demo_helper.duckdb'
# result = batch_query(
#     "SELECT order_id, status, amount FROM orders WHERE status = ?",
#     [['shipped'], ['pending'], ['delivered']],
#     db_path=DB_PATH,
# )
# print(result.sort_values('order_id'))


---
## Day 4 key concepts recap

| Concept | What to remember |
|---|---|
| `duckdb.connect()` | No arg → in-memory; string path → file-backed; `read_only=True` → concurrent reads |
| Parameterised queries | Use `?` placeholders + `[params]` list — never f-strings |
| Fetch methods | `fetchone()` scalar, `fetchall()` tuples, `fetchdf()` DataFrame, `fetch_arrow_table()` Arrow |
| `duckdb.sql()` | Module-level shortcut using implicit global connection — fine for exploration |
| `con.register()` | Expose Python list/dict/DataFrame as a queryable SQL table without copying data |
| DB-API cursor | `con.cursor()` gives a PEP-249-compliant cursor; `cur.description` lists column metadata |
| `with duckdb.connect()` | Context manager closes connection automatically — preferred over manual `.close()` |
| Helper function | `try/finally` guarantees cleanup; `params or []` prevents NoneType errors |

> **Tip:** Prefer parameterised queries (`?`) over f-strings for SQL — `duckdb.execute()` handles escaping correctly. It also avoids re-parsing the query plan when you loop over batches.

---
## What's next
**Day 5** → Zero-copy integration with Pandas, Polars, and Apache Arrow — query DataFrames directly without serialisation overhead.

Mark Day 4 complete in your [tracker](../index.html).
